In [ ]:
import math
import json
import random

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer

# устройство: gpu если доступен
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("устройство:", device)
if device.type == "cuda":
    print("видеокарта:", torch.cuda.get_device_name(0))

In [ ]:
# токенизатор
tokenizer = Tokenizer.from_file("Models/mrplip_17M_3/mrplip_17M_3_tokenizer.json")

sos_token_id = tokenizer.token_to_id("[SOS]")
eos_token_id = tokenizer.token_to_id("[EOS]")
pad_id = tokenizer.token_to_id("[PAD]")

MAX_LEN = 512
tokenizer.enable_padding(length=MAX_LEN)
tokenizer.enable_truncation(max_length=MAX_LEN)
vocab_size = tokenizer.get_vocab_size()
print("словарь:", vocab_size)

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, embed_size, heads):
        super().__init__()
        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads
        self.qkv = nn.Linear(embed_size, embed_size * 3)
        self.fc_out = nn.Linear(embed_size, embed_size)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        qkv = qkv.reshape(B, T, 3, self.heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = torch.tril(torch.ones(T, T)).to(x.device)
        scores = scores.masked_fill(mask == 0, float("-inf"))
        attn = torch.softmax(scores, dim=-1)
        out = attn @ v
        out = out.transpose(1, 2).reshape(B, T, C)
        return self.fc_out(out)


class Block(nn.Module):
    def __init__(self, embed_size, heads, dropout=0.1):
        super().__init__()
        self.attn = SelfAttention(embed_size, heads)
        self.ln1 = nn.LayerNorm(embed_size)
        self.ff = nn.Sequential(
            nn.Linear(embed_size, 4 * embed_size),
            nn.GELU(),
            nn.Linear(4 * embed_size, embed_size),
            nn.Dropout(dropout),
        )
        self.ln2 = nn.LayerNorm(embed_size)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


class MipleModel(nn.Module):
    def __init__(self, vocab_size, embed_size, block_size, heads, decisions_count, emotions_count, market_features, sentiments_count, tactics_count, n_layers, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.token_embed = nn.Embedding(vocab_size, embed_size)
        self.pos_embed = nn.Embedding(block_size, embed_size)

        self.blocks = nn.Sequential(*[Block(embed_size, heads, dropout) for _ in range(n_layers)])

        self.market_proj = nn.Linear(market_features, embed_size)
        self.sent_proj = nn.Linear(sentiments_count, embed_size)

        combined_input_size = (embed_size * 3) + emotions_count + sentiments_count

        self.buy_decision = nn.Sequential(
            nn.Linear(combined_input_size, embed_size),
            nn.ReLU(),
            nn.Linear(embed_size, embed_size),
            nn.Linear(embed_size, decisions_count),
        )

        self.emotions_choicer = nn.Linear(combined_input_size, emotions_count)
        self.tactic_choicer = nn.Linear(combined_input_size, tactics_count)

        # выбор акции: к каждой акции клеим личность и считаем балл
        self.stock_selector = nn.Sequential(
            nn.Linear(embed_size * 2, embed_size),
            nn.ReLU(),
            nn.Linear(embed_size, 1),
        )

        self.ln_f = nn.LayerNorm(embed_size)
        self.fc = nn.Linear(embed_size, vocab_size)

    def forward(self, market_seq, prompt_seq, news_seq, current_emotions, sentiments_config):
        B, T = prompt_seq.shape

        token_embeddings = self.token_embed(prompt_seq)
        position_embeddings = self.pos_embed(torch.arange(T, device=prompt_seq.device))
        x = token_embeddings + position_embeddings
        x = self.blocks(x)

        market_emb = self.market_proj(market_seq)
        news_emb = self.token_embed(news_seq)

        market_emb_mean = torch.mean(market_emb, dim=1)
        news_emb_mean = torch.mean(news_emb, dim=1)
        x_mean = torch.mean(x, dim=1)

        sent_emb = self.sent_proj(sentiments_config)

        combined = torch.cat((x_mean, market_emb_mean, news_emb_mean, current_emotions, sentiments_config), dim=1)

        trade_logits = self.buy_decision(combined)
        new_emotions = self.emotions_choicer(combined)
        tactic_logits = self.tactic_choicer(combined)

        n_stocks = market_emb.shape[1]
        sent_rep = sent_emb.unsqueeze(1).expand(-1, n_stocks, -1)
        stock_feat = torch.cat((market_emb, sent_rep), dim=2)
        stock_logits = self.stock_selector(stock_feat).squeeze(-1)

        x_norm = self.ln_f(x)
        text_logits = self.fc(x_norm)

        return trade_logits, new_emotions, tactic_logits, stock_logits, text_logits

In [ ]:
THEMES = ["технологии", "медицина", "финансы", "энергетика", "товары", "недвижимость",
          "сырьё", "промышленность", "связь", "спорт", "развлечения", "транспорт", "агро"]
DIR_MAP = {"рост": 1.0, "флэт": 0.0, "падение": -1.0}
SENTIMENTS = ["оптимист", "рискованный", "систематичный", "консерватист", "интроверт", "убежденный", "интуитивный"]


class FineTuningDataset(Dataset):
    def __init__(self, tokenizer, samples, max_len=512, max_stocks=8):
        super().__init__()
        self.tokenizer = tokenizer
        self.samples = samples
        self.max_len = max_len
        self.max_stocks = max_stocks
        # словарь тактик строим прямо из датасета
        tactics = sorted({s["tactic"] for s in samples})
        self.tactic2idx = {t: i for i, t in enumerate(tactics)}

    def __len__(self):
        return len(self.samples)

    def encode_market(self, stocks_states):
        names = list(stocks_states.keys())
        rows = []
        for name in names:
            sphere, price, dir_v, vol = stocks_states[name]
            sphere_idx = THEMES.index(sphere) / len(THEMES) if sphere in THEMES else 0.0
            rows.append([sphere_idx, float(price), DIR_MAP.get(dir_v, 0.0), float(vol)])
        while len(rows) < self.max_stocks:
            rows.append([0.0, 0.0, 0.0, 0.0])
        return torch.tensor(rows[:self.max_stocks], dtype=torch.float), names

    def __getitem__(self, index):
        s = self.samples[index]

        context = " ".join(s["messages"])
        text = f"{context} [SOS] {s['answer']} [EOS]"
        prompt_ids = torch.tensor(self.tokenizer.encode(text).ids, dtype=torch.long)

        news = " ".join(s["events"])
        news_ids = torch.tensor(self.tokenizer.encode(news).ids, dtype=torch.long)

        market, names = self.encode_market(s["stocks_states"])

        emotions = torch.tensor(s["emotions"], dtype=torch.float)
        sentiments = torch.tensor(s["basic_sentiments_parameters"], dtype=torch.float)
        decision = torch.tensor(s["decisions"].index(1), dtype=torch.long)
        stock_target = torch.tensor(names.index(s["target_stock"]) if s["target_stock"] in names else 0, dtype=torch.long)
        tactic_target = torch.tensor(self.tactic2idx[s["tactic"]], dtype=torch.long)

        return market, prompt_ids, news_ids, emotions, sentiments, decision, stock_target, tactic_target


class TextDataset(Dataset):
    def __init__(self, tokenizer, samples, use_messages=True):
        super().__init__()
        self.tokenizer = tokenizer
        # вытаскиваем сырой текст из основного датасета (реплики + контекст)
        seen = set()
        self.texts = []
        for s in samples:
            chunk = list(s["messages"]) if use_messages else []
            chunk.append(s["answer"])
            for t in chunk:
                t = t.strip()
                if t and t not in seen:
                    seen.add(t)
                    self.texts.append(t)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        text = f"[SOS] {self.texts[index]} [EOS]"
        return torch.tensor(self.tokenizer.encode(text).ids, dtype=torch.long)

In [ ]:
with open("MipleDataset.json", encoding="utf-8") as f:
    samples = json.load(f)

embed_size = 512
block_size = 512
heads = 8
decisions_count = 3
emotions_count = 3
market_features = 4
sentiments_count = 7
stocks_count = 8
n_layers = 4

dataset = FineTuningDataset(tokenizer, samples, max_len=MAX_LEN, max_stocks=stocks_count)
tactics_count = len(dataset.tactic2idx)

# на gpu выгодно пиннить память
pin = device.type == "cuda"
loader = DataLoader(dataset, batch_size=8, shuffle=True, pin_memory=pin)

text_dataset = TextDataset(tokenizer, samples)
text_loader = DataLoader(text_dataset, batch_size=8, shuffle=True, pin_memory=pin)

print("сэмплов:", len(dataset), "текстов:", len(text_dataset), "тактик:", tactics_count)


def make_persona(active, base=0.1, strong=0.9):
    # тут настройка сентиментов
    vec = torch.full((len(SENTIMENTS),), base)
    for s in active:
        vec[SENTIMENTS.index(s)] = strong
    return vec.unsqueeze(0)

In [ ]:
# модель сразу на устройство
model = MipleModel(vocab_size, embed_size, block_size, heads, decisions_count, emotions_count, market_features, sentiments_count, tactics_count, n_layers).to(device)
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

ce = nn.CrossEntropyLoss(ignore_index=pad_id)
mse = nn.MSELoss()

# веса классов решений
dec_counts = [0, 0, 0]
for s in samples:
    dec_counts[s["decisions"].index(1)] += 1
dec_w = torch.tensor([len(samples) / (3 * c) if c else 0.0 for c in dec_counts]).to(device)
ce_trade = nn.CrossEntropyLoss(weight=dec_w)

# mixed precision только на gpu
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

In [ ]:
history = {"total": [], "text": [], "trade": [], "stock": [], "tactic": [], "emo": []}

# претрейн только по тексту
pretrain_epochs = 2
model.train()
for epoch in range(pretrain_epochs):
    last = 0.0
    for prompt_ids in text_loader:
        prompt_ids = prompt_ids.to(device, non_blocking=True)
        B = prompt_ids.shape[0]
        z_market = torch.zeros(B, stocks_count, market_features, device=device)
        z_news = torch.full((B, 1), sos_token_id, dtype=torch.long, device=device)
        z_emo = torch.zeros(B, emotions_count, device=device)
        z_sent = torch.zeros(B, sentiments_count, device=device)

        with torch.autocast(device_type=device.type, enabled=use_amp):
            _, _, _, _, text_logits = model(z_market, prompt_ids, z_news, z_emo, z_sent)
            text_loss = ce(text_logits[:, :-1, :].reshape(-1, vocab_size), prompt_ids[:, 1:].reshape(-1))

        optimizer.zero_grad()
        scaler.scale(text_loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        last = text_loss.item()
    print(f"[pretrain {epoch+1}] text {last:.3f}")


# основное обучение по всем головам
epochs = 15
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
for epoch in range(epochs):
    for market, prompt_ids, news_ids, emotions, sentiments, decision, stock_target, tactic_target in loader:
        market = market.to(device, non_blocking=True)
        prompt_ids = prompt_ids.to(device, non_blocking=True)
        news_ids = news_ids.to(device, non_blocking=True)
        emotions = emotions.to(device, non_blocking=True)
        sentiments = sentiments.to(device, non_blocking=True)
        decision = decision.to(device, non_blocking=True)
        stock_target = stock_target.to(device, non_blocking=True)
        tactic_target = tactic_target.to(device, non_blocking=True)

        # эмоций на входе нет
        zero_emo = torch.zeros_like(emotions)
        with torch.autocast(device_type=device.type, enabled=use_amp):
            trade_logits, new_emotions, tactic_logits, stock_logits, text_logits = model(market, prompt_ids, news_ids, zero_emo, sentiments)

            text_loss = ce(text_logits[:, :-1, :].reshape(-1, vocab_size), prompt_ids[:, 1:].reshape(-1))
            trade_loss = ce_trade(trade_logits, decision)
            stock_loss = ce(stock_logits, stock_target)
            tactic_loss = ce(tactic_logits, tactic_target)
            emo_loss = mse(new_emotions, emotions)

            loss = 0.3 * text_loss + 1.5 * trade_loss + stock_loss + tactic_loss + 0.5 * emo_loss

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        history["total"].append(loss.item())
        history["text"].append(text_loss.item())
        history["trade"].append(trade_loss.item())
        history["stock"].append(stock_loss.item())
        history["tactic"].append(tactic_loss.item())
        history["emo"].append(emo_loss.item())

    scheduler.step()
    print(f"[epoch {epoch}] total {history['total'][-1]:.3f} | text {history['text'][-1]:.3f} trade {history['trade'][-1]:.3f} stock {history['stock'][-1]:.3f} tactic {history['tactic'][-1]:.3f} emo {history['emo'][-1]:.3f}")

In [ ]:
# сохраняем веса на проц, чтобы все грузилось на любом устройстве
cpu_state = {k: v.cpu() for k, v in model.state_dict().items()}
torch.save(cpu_state, "Models/mrplip_17M_3/mrplip_17M_3_gpu.pth")
print("сохранено в Models/mrplip_17M_3/mrplip_17M_3_gpu.pth")

In [ ]:
idx2tactic = {i: t for t, i in dataset.tactic2idx.items()}


def strip_pad(ids):
    return [i for i in ids if i != pad_id]


@torch.no_grad()
def predict(stocks_states, events, persona, messages=None, max_new=40, temperature=0.9, greedy_text=False):
    model.eval()

    names = list(stocks_states.keys())
    market, _ = dataset.encode_market(stocks_states)
    market = market.unsqueeze(0).to(device)

    news_ids = strip_pad(tokenizer.encode(" ".join(events)).ids) or [sos_token_id]
    news_seq = torch.tensor([news_ids], dtype=torch.long, device=device)

    emotions = torch.zeros(1, 3, device=device)
    sentiments = persona.to(device)

    context = " ".join(messages or []).strip()
    prompt_text = f"{context} [SOS]".strip()
    prompt = torch.tensor([strip_pad(tokenizer.encode(prompt_text).ids)], dtype=torch.long, device=device)

    trade_logits, new_emotions, tactic_logits, stock_logits, _ = model(market, prompt, news_seq, emotions, sentiments)
    decision = ["buy", "sell", "hold"][trade_logits.argmax(-1).item()]
    tactic = idx2tactic[tactic_logits.argmax(-1).item()]
    chosen_stock = names[stock_logits[0, :len(names)].argmax().item()]
    emo = new_emotions.squeeze(0).tolist()

    generated = []
    for _ in range(max_new):
        if prompt.shape[1] >= model.block_size:
            break
        _, _, _, _, text_logits = model(market, prompt, news_seq, emotions, sentiments)
        logits = text_logits[0, -1]
        if greedy_text:
            nxt = logits.argmax().item()
        else:
            probs = torch.softmax(logits / temperature, dim=-1)
            nxt = torch.multinomial(probs, 1).item()
        if nxt == eos_token_id:
            break
        generated.append(nxt)
        prompt = torch.cat([prompt, torch.tensor([[nxt]], device=device)], dim=1)

    return {
        "answer": tokenizer.decode(generated),
        "target_stock": chosen_stock,
        "decision": decision,
        "tactic": tactic,
        "emotions": {"angry": round(emo[0], 2), "sadness": round(emo[1], 2), "joy": round(emo[2], 2)},
    }


s = samples[0]
print(predict(s["stocks_states"], s["events"], make_persona(["систематичный"]), s["messages"]))